In [6]:
import tkinter as tk
from tkinter import filedialog
from PIL import Image
from PIL.TiffTags import TAGS
from datetime import datetime

def select_and_extract_metadata():
    # Create a Tkinter root window and hide it
    root = tk.Tk()
    root.withdraw()

    # Open file dialog to select a .tif file
    file_path = filedialog.askopenfilename(filetypes=[("TIFF files", "*.tif")])
    if not file_path:
        print("No file selected.")
        return

    try:
        with Image.open(file_path) as img:
            metadata = {TAGS.get(key, key): img.tag[key] for key in img.tag.keys()}
            print("Metadata extracted:")
            for tag, value in metadata.items():
                print(f"{tag}: {value}")
    except Exception as e:
        print(f"Error reading file: {e}")
        # Assuming 'metadata' already exists
    raw_datetime = metadata.get("DateTime")
    if raw_datetime:
        dt_str = raw_datetime[0].split('\x00')[0]  # remove null characters
        try:
            dt_obj = datetime.strptime(dt_str, "%Y:%m:%d %H:%M:%S")
            print("Parsed creation time:", dt_obj)
        except ValueError as e:
            print("Could not parse datetime:", e)
    else:
        print("No DateTime tag found.")
select_and_extract_metadata()


Metadata extracted:
ImageWidth: (981,)
ImageLength: (1043,)
BitsPerSample: (32,)
Compression: (1,)
36865: (0,)
36866: (0,)
PhotometricInterpretation: (1,)
DateTimeOriginal: (1.0,)
DateTimeDigitized: (0.0,)
37120: (0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)
36876: (0.0,)
36877: (0.0,)
ImageDescription: ('# Pixel_size 172e-6 m x 172e-6 m\r\n# Silicon sensor, thickness 0.000450\r\n# Exposure_time 1.0000000 s\r\n# Exposure_period 1.0000000 s\r\n# Tau = 0 s\r\n# Count_cutoff 1409688 counts\r\n# Threshold_setting: 4024 eV\r\n# Gain_setting: autog (vrf = 1.000)\r\n# N_excluded_pixels = 31\r\n# Excluded_pixels: badpix_mask.tif\r\n# Flat_field: FF_p10-0199_E8048_T4024_vrf_m0p100.tif\r\n# Trim_file: p10-0199_E8048_T4024.bin\r\n# Image_path: /xmasdata/2024/printz_Apr2024/MAPI_25pct_APA_S3_30_tub/\r\n# Comment: 7.00000 10.7870 10.0000 5373.0000\r\n# Ratecorr_lut_directory: ContinuousStandard_v1.1\r\n# Retrigger_mode: 1\r\n',)
36878: (0.0,)
Model: ('PILATUS3 X_1M, SN 10-0199\x00\x0

In [2]:
import numpy as np
from mayavi import mlab

# Generate asymmetric 3D test data
x, y, z = np.mgrid[0:50, 0:50, 0:50]
data = (
    np.exp(-((x - 30) ** 2 + (y - 20) ** 2 + (z - 10) ** 2) / 100) +
    0.5 * np.sin(x / 10) * np.cos(y / 15) * np.exp(-(z - 25) ** 2 / 50)
)

# Create Mayavi figure
mlab.figure(bgcolor=(1, 1, 1), size=(800, 600))

# 3D isosurface
mlab.contour3d(data, contours=5, opacity=0.4, colormap='viridis')

# Add slices projected on orthogonal planes
src = mlab.pipeline.scalar_field(data)
src.update_image_data = True

# Slice at x = 25
mlab.pipeline.image_plane_widget(src,
                                 plane_orientation='x_axes',
                                 slice_index=25,
                                 colormap='viridis')

# Slice at y = 20
mlab.pipeline.image_plane_widget(src,
                                 plane_orientation='y_axes',
                                 slice_index=20,
                                 colormap='viridis')

# Slice at z = 15
mlab.pipeline.image_plane_widget(src,
                                 plane_orientation='z_axes',
                                 slice_index=15,
                                 colormap='viridis')

# Axes and labels
mlab.axes(xlabel='t1', ylabel='t2', zlabel='ω', color=(0, 0, 0))
mlab.outline(color=(0, 0, 0))

mlab.view(azimuth=45, elevation=70, distance=200)
mlab.show()


In [4]:
import numpy as np
from mayavi import mlab

# Create asymmetric test data
x, y, z = np.mgrid[0:50, 0:50, 0:50]
data = (
    np.exp(-((x - 35)**2 + (y - 10)**2 + (z - 20)**2) / 100) +
    0.4 * np.sin(x / 10) * np.cos(z / 15) * np.exp(-(y - 25)**2 / 70)
)

# Set up figure
mlab.figure(bgcolor=(1, 1, 1), size=(800, 700))

# 3D isosurface
src = mlab.pipeline.scalar_field(data)
src.update_image_data = True
contour = mlab.pipeline.contour(src)
contour.filter.contours = [0.2, 0.4, 0.6, 0.8]
contour = mlab.pipeline.surface(contour, colormap='viridis', opacity=0.4)


# Static orthogonal projections (image_plane_widget with `enable=False`)
# XY projection (z = 0)
ipw_xy = mlab.pipeline.image_plane_widget(src,
    plane_orientation='z_axes',
    slice_index=0,
    colormap='viridis',
)

# XZ projection (y = 0)
ipw_xz = mlab.pipeline.image_plane_widget(src,
    plane_orientation='y_axes',
    slice_index=0,
    colormap='viridis',
)

# YZ projection (x = max index)
ipw_yz = mlab.pipeline.image_plane_widget(src,
    plane_orientation='x_axes',
    slice_index=data.shape[0] - 1,
    colormap='viridis',
)

# Add axes and outline
mlab.outline(src, color=(0, 0, 0))
mlab.axes(xlabel='t1', ylabel='t2', zlabel='ω', color=(0, 0, 0), nb_labels=5)

# Adjust camera
mlab.view(azimuth=45, elevation=70, distance=200)
mlab.show()


In [12]:
import pandas as pd
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog

logFile = filedialog.askopenfilename(title="Select log file", filetypes=[("Text files", "*.txt")])
names=np.array(['Time of Day', 'Time', 'Image Counts', 'Pyrometer','Spin_Motor', 'Dispense X', 'Dispense Z', 'Gas Quenching', 'Sine' , 'BK Set Amps', 'BK Set Volts', 'BK Amps', 'BK Volts', 'BK Power', '2D Image', 'Spectrometer'])

with open(logFile) as f:
            for i, l in enumerate(f):
                if l.startswith('DATA'):
                    header = i+1
                    break
logData = pd.read_csv(logFile, sep='\t',  header = header)
print("Column names of logData:", logData.columns.tolist())
print(logData.head())
logData = logData.rename(columns={
    'Time (s)': 'Time',
    'Spin Motor': 'Spin_Motor',
})
logSelection = ['Time', 'Pyrometer', 'Spin_Motor', 'Dispense X']
logDataSelect = logData[logSelection]
logDataSelect.Spin_Motor.to_clipboard()

Column names of logData: ['Time of Day', 'Time (s)', 'Image Counts', 'Pyrometer', 'Spin Motor', 'BK Set Amps', 'BK Set Volts', 'BK Amps', 'BK Volts', 'BK Watts', 'BK Resistance', 'Dispense X', 'Dispense Z', 'Gas Quenching', 'Crystal 1 Encoder', 'Crystal 2 Encoder', 'Mono Energy', 'Beam Current', 'SR Energy', 'Counter 0', 'Counter 1', 'Counter 2', 'Counter 3', 'Counter Gate', 'Vert Centroid', 'Horz Centroid', 'Vert FWHM', 'Horz FWHM', 'M1 Bend 1', 'M1 Bend 2', 'Horizontal KB Translation', 'Horizontal KB Pitch', 'Horizontal KB Bend 1', 'Horizontal KB Bend 2', 'Horizontal KB Roll', 'Horizontal KB Z', 'Vertical KB Translation', 'Vertical KB Pitch', 'Vertical KB Bend 1', 'Vertical KB Bend 2', 'Vertical KB Roll', 'TC KB Horz', 'TC Hutch 1', 'TC Tank 1', 'TC Hutch 2', 'TC Tank 2', 'TC Hutch 3', 'TC Tank 3', 'TC Hutch 4', 'Strain Gauge Channel 0', 'Strain Gauge Channel 1', 'AI Channel 0', 'AI Channel 1', 'AI Channel 2', 'Load Stage Strain Gauge', 'Load Stage LVDT', 'Centroid Intensity', 'Video

In [ ]:

import numpy as np
from itertools import product

a = 6.31  # lattice constant in Å (cubic MAPbI3)

# Generate all hkl combinations excluding (0, 0, 0)
hkls = [
    (h, k, l)
    for h, k, l in product(range(3), repeat=3)
    if (h, k, l) != (0, 0, 0)
]

# Compute d-spacing
d_spacing_dict = {}
for hkl in hkls:
    h2k2l2 = hkl[0]**2 + hkl[1]**2 + hkl[2]**2
    if h2k2l2 == 0:
        continue
    d = a / np.sqrt(h2k2l2)
    d = round(d, 4)  # round to 4 decimals for uniqueness
    if d not in d_spacing_dict:
        d_spacing_dict[d] = [hkl]
    else:
        d_spacing_dict[d].append(hkl)

# Sort and display
for d in sorted(d_spacing_dict.keys(), reverse=True):
    hkls = d_spacing_dict[d]
    print(f"hkl:{hkls}, d: {d:.4f}")


6.3100
4.4618
3.6431
3.1550
2.8219
2.5760
2.2309
2.1033
1.8215
